In [1]:
import warnings
warnings.filterwarnings('ignore')

# Importing necessary libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau
import plotly.graph_objects as go
import plotly.express as px

2025-05-21 03:08:19.414282: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747796899.660299      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747796899.728080      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# Paths to the dataset
dataset_path = "/kaggle/input/animal-image-dataset-90-different-animals/animals/animals"
labels_file = "/kaggle/input/animal-image-dataset-90-different-animals/name of the animals.txt"

In [3]:
# Reading labels
with open(labels_file, 'r') as f:
    animal_names = f.read().split('\n')

In [4]:
animal_names[:10]

['antelope',
 'badger',
 'bat',
 'bear',
 'bee',
 'beetle',
 'bison',
 'boar',
 'butterfly',
 'cat']

In [5]:
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.resize(image, (180, 180))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image / 255.0
    return image

In [6]:
data = []
labels = []

for animal in animal_names:
    animal_dir = os.path.join(dataset_path, animal)
    for img_name in os.listdir(animal_dir):
        img_path = os.path.join(animal_dir, img_name)
        data.append(preprocess_image(img_path))
        labels.append(animal)

data = np.array(data)
labels = np.array(labels)

In [7]:
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

In [9]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [10]:
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(180, 180, 3))
base_model.trainable = False  # Giai đoạn 1: Freeze toàn bộ

I0000 00:00:1747796981.161663      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [11]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(len(animal_names), activation='softmax')
])

In [12]:
# Compile and train (Giai đoạn 1)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

In [13]:
print("==== GIAI ĐOẠN 1: Huấn luyện head ====")
history1 = model.fit(datagen.flow(X_train, y_train, batch_size=32), 
                     epochs=10, 
                     validation_data=(X_test, y_test))

==== GIAI ĐOẠN 1: Huấn luyện head ====
Epoch 1/10


I0000 00:00:1747796999.103363      93 service.cc:148] XLA service 0x7ef3d845a900 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1747796999.104482      93 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1747797000.967714      93 cuda_dnn.cc:529] Loaded cuDNN version 90300


  1/135 ━━━━━━━━━━━━━━━━━━━━ 41:16 18s/step - accuracy: 0.0312 - loss: 5.9098

I0000 00:00:1747797006.809398      93 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


135/135 ━━━━━━━━━━━━━━━━━━━━ 59s 299ms/step - accuracy: 0.2083 - loss: 4.0231 - val_accuracy: 0.7185 - val_loss: 1.1526
Epoch 2/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 28s 209ms/step - accuracy: 0.5990 - loss: 1.6278 - val_accuracy: 0.7630 - val_loss: 0.8720
Epoch 3/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 28s 207ms/step - accuracy: 0.6452 - loss: 1.3723 - val_accuracy: 0.7954 - val_loss: 0.7915
Epoch 4/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 28s 208ms/step - accuracy: 0.6760 - loss: 1.1742 - val_accuracy: 0.7981 - val_loss: 0.7382
Epoch 5/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 28s 208ms/step - accuracy: 0.6997 - loss: 1.0644 - val_accuracy: 0.7954 - val_loss: 0.7180
Epoch 6/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 28s 209ms/step - accuracy: 0.7164 - loss: 0.9966 - val_accuracy: 0.8083 - val_loss: 0.6790
Epoch 7/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 28s 210ms/step - accuracy: 0.7364 - loss: 0.9831 - val_accuracy: 0.8204 - val_loss: 0.6439
Epoch 8/10
135/135 ━━━━━━━━━━━━━━━━━━━━ 29s 212ms/step - accuracy: 0.7291 - loss: 0.9617 - val

In [14]:
# Giai đoạn 2: Fine-tune 50 layer cuối
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False  # Chỉ mở 50 lớp cuối

# Compile lại với learning rate nhỏ hơn
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

In [15]:
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)

In [16]:
print("==== GIAI ĐOẠN 2: Fine-tune 50 lớp cuối của InceptionV3 ====")
history2 = model.fit(datagen.flow(X_train, y_train, batch_size=32), 
                     epochs=40, 
                     validation_data=(X_test, y_test), 
                     callbacks=[lr_scheduler])

==== GIAI ĐOẠN 2: Fine-tune 50 lớp cuối của InceptionV3 ====
Epoch 1/40
135/135 ━━━━━━━━━━━━━━━━━━━━ 64s 289ms/step - accuracy: 0.7351 - loss: 1.1128 - val_accuracy: 0.8380 - val_loss: 0.5820 - learning_rate: 1.0000e-04
Epoch 2/40
135/135 ━━━━━━━━━━━━━━━━━━━━ 29s 211ms/step - accuracy: 0.8048 - loss: 0.6922 - val_accuracy: 0.8444 - val_loss: 0.5698 - learning_rate: 1.0000e-04
Epoch 3/40
135/135 ━━━━━━━━━━━━━━━━━━━━ 29s 212ms/step - accuracy: 0.8402 - loss: 0.5908 - val_accuracy: 0.8611 - val_loss: 0.5259 - learning_rate: 1.0000e-04
Epoch 4/40
135/135 ━━━━━━━━━━━━━━━━━━━━ 29s 212ms/step - accuracy: 0.8581 - loss: 0.5173 - val_accuracy: 0.8556 - val_loss: 0.5118 - learning_rate: 1.0000e-04
Epoch 5/40
135/135 ━━━━━━━━━━━━━━━━━━━━ 29s 213ms/step - accuracy: 0.8908 - loss: 0.4210 - val_accuracy: 0.8556 - val_loss: 0.5170 - learning_rate: 1.0000e-04
Epoch 6/40
135/135 ━━━━━━━━━━━━━━━━━━━━ 29s 211ms/step - accuracy: 0.8910 - loss: 0.3810 - val_accuracy: 0.8722 - val_loss: 0.4972 - learning_ra

In [17]:
history = {
    'accuracy': history1.history['accuracy'] + history2.history['accuracy'],
    'val_accuracy': history1.history['val_accuracy'] + history2.history['val_accuracy'],
    'loss': history1.history['loss'] + history2.history['loss'],
    'val_loss': history1.history['val_loss'] + history2.history['val_loss'],
}

In [19]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(1, len(history['accuracy'])+1)), y=history['accuracy'], mode='lines+markers', name='Training Accuracy'))
fig.add_trace(go.Scatter(x=list(range(1, len(history['val_accuracy'])+1)), y=history['val_accuracy'], mode='lines+markers', name='Validation Accuracy'))
fig.update_layout(title='Model Accuracy', xaxis_title='Epoch', yaxis_title='Accuracy', title_font_size=24, title_x=0.5, xaxis_title_font_size=18, yaxis_title_font_size=18, font=dict(family="Arial, sans-serif", size=14), template='plotly_dark')
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(1, len(history['loss'])+1)), y=history['loss'], mode='lines+markers', name='Training Loss'))
fig.add_trace(go.Scatter(x=list(range(1, len(history['val_loss'])+1)), y=history['val_loss'], mode='lines+markers', name='Validation Loss'))
fig.update_layout(title='Model Loss', xaxis_title='Epoch', yaxis_title='Loss', title_font_size=24, title_x=0.5, xaxis_title_font_size=18, yaxis_title_font_size=18, font=dict(family="Arial, sans-serif", size=14), template='plotly_dark')
fig.show()

In [20]:
# Evaluate model
y_pred = np.argmax(model.predict(X_test), axis=-1)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

34/34 ━━━━━━━━━━━━━━━━━━━━ 14s 237ms/step
                precision    recall  f1-score   support

      antelope       0.64      0.64      0.64        11
        badger       1.00      1.00      1.00        19
           bat       0.85      1.00      0.92        11
          bear       1.00      0.85      0.92        13
           bee       0.89      1.00      0.94        16
        beetle       0.92      1.00      0.96        11
         bison       0.91      0.83      0.87        12
          boar       0.87      0.81      0.84        16
     butterfly       0.83      0.83      0.83        12
           cat       0.85      0.85      0.85        13
   caterpillar       0.89      0.73      0.80        11
    chimpanzee       1.00      0.92      0.96        12
     cockroach       1.00      1.00      1.00        15
           cow       0.82      0.60      0.69        15
        coyote       0.77      0.91      0.83        11
          crab       1.00      1.00      1.00        10
     